# Chấm test REINFORCE và RLOO để so với GRPO (ck500)

**Câu hỏi:** cùng S1, cùng 1.000 câu nhắc, cùng 500 bước, cùng thưởng SPICE, chỉ đổi cách trừ mốc của thuật toán RL thì exec khác ck500 thế nào. Hai adapter đã train xong, ở đây chỉ sinh câu và chấm trên tập kiểm.

- **REINFORCE + mốc EMA** chạy trước: thuật toán cổ điển, mốc là trung bình trượt toàn cục.
- **RLOO** chạy sau: mốc là trung bình các mẫu còn lại trong nhóm, họ gần GRPO nhất.

**Cách chạy**
1. Runtime → Change runtime type → **T4 GPU**. Dùng T4 để cùng phần cứng và fp16 với mọi lượt đã chấm (L4 chạy bf16 có thể lệch ±1 bước khi chấm).
2. Bấm **Run all**. Ô 2 hỏi khoá Kaggle của tài khoản `uyenle0712` (hoặc đặt sẵn ở Colab Secrets tên `KAGGLE_TOKEN`).
3. Giữ tab mở, máy không ngủ. Ước tính [theo các lượt test trước trên T4] **4–5 giờ mỗi lượt**. REINFORCE xong là có số ngay, RLOO không kịp thì dừng cũng không mất gì.

**Mất máy:** bấm Run all lại, ô 3 chạy tiếp từ bản lưu trên Drive (`MyDrive/thesis/luoi_test_colab/`), chép mỗi 5 phút.

**Đừng** bấm chạy lại ô 3 khi tiến trình cũ còn sống: Stop ô không giết tiến trình con.

## Ô 1. Môi trường

In [ ]:
import os, sys, subprocess, time, json, glob, shutil, hashlib, threading, zipfile
from google.colab import drive
T0 = time.time()
drive.mount("/content/drive")
DR = "/content/drive/MyDrive/thesis/luoi_test_colab"; os.makedirs(DR, exist_ok=True)
W = "/content/w"; os.makedirs(W, exist_ok=True)

def sh(c, duoi=1500):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-duoi:], flush=True)
    return r.returncode

ten = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", ten, flush=True)
assert ten, "DUNG: chua bat GPU"
if "T4" not in ten:
    print("⚠️ KHONG PHAI T4: cac luot da cham deu tren T4 fp16. Co the lech ±1 buoc, phai khai khi bao so.", flush=True)
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate bitsandbytes 'kaggle==2.2.4' scipy 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1")
assert sh(f"""{sys.executable} -c "import trl, transformers, peft, torch; print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__, torch.cuda.get_device_name(0)); assert (trl.__version__, transformers.__version__, peft.__version__) == ('0.29.1', '5.20.2', '0.21.2'), 'lech phien ban'" """) == 0, "DUNG: moi truong loi"
print("✅ moi truong", flush=True)

## Ô 2. Tải dữ liệu (khoảng 5 phút)
Khoá chỉ ghi vào `/root/.kaggle`, không in ra, không lưu vào Drive hay notebook.

In [ ]:
import getpass, stat
KD = "/root/.kaggle"; os.makedirs(KD, exist_ok=True); TK = f"{KD}/access_token"
if not os.path.exists(TK):
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get("KAGGLE_TOKEN")
    except Exception:
        pass
    tok = tok or getpass.getpass("Khoa Kaggle cua uyenle0712 (KGAT_...): ")
    with open(TK, "w") as f: f.write(tok.strip())
    os.chmod(TK, stat.S_IRUSR | stat.S_IWUSR); del tok
ENV_K = {**os.environ, "KAGGLE_CONFIG_DIR": KD, "KAGGLE_API_TOKEN": TK}

def tai(ds, dich):
    if os.path.exists(dich + "/.xong"): print("da co", ds, flush=True); return
    os.makedirs(dich, exist_ok=True)
    r = subprocess.run(["kaggle", "datasets", "download", ds, "-p", dich, "--unzip"], env=ENV_K, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-300:], flush=True); assert r.returncode == 0, f"DUNG: khong tai duoc {ds}"
    for z in glob.glob(dich + "/*.zip"):
        d = z[:-4]; os.makedirs(d, exist_ok=True); shutil.unpack_archive(z, d); os.remove(z)
    open(dich + "/.xong", "w").close()

KIT, DSK = f"{W}/kit", f"{W}/thesis_score"
tai("uyenle0712/sft-colab-kit", KIT)
tai("uyenle0712/thesis-score", DSK)
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
assert md5(f"{KIT}/grpo_spice.py") == "07ea87b6d156d1faa391a4274dffd7cf", "DUNG: grpo_spice.py lech"
assert md5(f"{KIT}/gen_test_grpo.py") == "b18ec1aa3fd037ddffb9411db63c2a3f", "DUNG: gen_test_grpo.py lech"
assert sum(1 for _ in open(f"{KIT}/preds_s1_seed101.jsonl")) == 6958
# thu tu chay; muon them SFT tiep thi them ("sft_tiep_101", "SFT tiep", "sft_final", "113e2a3e6a0781fd5121acda99962d1b") vao cuoi
LUOT = [("reinforce_spice_101", "REINFORCE + SPICE", "reinforce_spice_101", "e0e9c51585f416337a25c52f6e7c8505"),
        ("rloo_spice_101", "RLOO + SPICE", "rloo_spice_101", "0efff3eb11f96370097eeafa9410e8cc")]
for r, _, d, h in LUOT:
    assert md5(f"{KIT}/{d}/adapter_model.safetensors") == h, f"DUNG: adapter {r} lech"
    if os.path.exists(f"{KIT}/{d}/trainer_state.json"):
        assert json.load(open(f"{KIT}/{d}/trainer_state.json"))["global_step"] == 500, f"DUNG: {r} khong phai buoc 500"
for f in ("grpo_spice.py", "gen_test_grpo.py", "build_branch_data.py", "tai_dung_cham.py", "doc_luot.py"):
    shutil.copy(f"{KIT}/{f}", f"{W}/{f}")
t = glob.glob(f"{DSK}/**/test_ac/test.jsonl", recursive=True); assert len(t) == 1, t
TA = os.path.dirname(t[0])
assert md5(f"{TA}/test.jsonl") == "da58299ee551a926a21cbecb5232bf78", "DUNG: test.jsonl lech"
assert len(glob.glob(f"{TA}/images/*.png")) >= 4463 and os.path.exists(f"{TA}/ocr.jsonl"), "DUNG: thieu anh hoac OCR"
WS = os.path.dirname(os.path.dirname(os.path.dirname(TA)))     # chua harness/
assert os.path.exists(f"{WS}/harness/score_run.py"), WS
NGUON = sorted(glob.glob(f"{KIT}/raw_*.jsonl")); assert len(NGUON) == 8, NGUON
print("✅ du lieu ·", TA, "·", len(NGUON), "tep tho de dung lai ket qua cham", flush=True)

## Ô 3. Sinh câu và chấm (foreground)
Hai phép kiểm chạy trước trong khoảng 10 phút, sai là dừng ngay:
- **Bộ chấm:** 10 câu S1 chấm lại phải trùng tệp thô cũ, lệch tối đa 1 câu.
- **Kiểm hoà:** S1 hoà sinh lại 20 câu, trùng S1 công bố ít nhất 16/20.

Mỗi lượt xong sẽ in ngay số so với S1 và ck500, và lưu `<lượt>.zip` lên Drive. Tự kiểm ck500 − S1 phải ra +1,55 [+0,80; +2,33] và +28/+41 thì số mới dùng được.

In [ ]:
MERGED, BUNDLE = f"{W}/s1_merged", KIT
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1", "PYTORCH_ALLOC_CONF": "expandable_segments:True", "CUDA_VISIBLE_DEVICES": "0"}
TA_ARGS = ["--recs", f"{TA}/test.jsonl", "--ocr", f"{TA}/ocr.jsonl", "--images", f"{TA}/images", "--tap-only", "--no-q4"]
dem = lambda p: sum(1 for _ in open(p, errors="ignore")) if os.path.exists(p) else 0
CUR = []

def chep_mot(p):
    q = f"{DR}/{os.path.basename(p)}"
    if os.path.exists(p) and dem(p) >= dem(q):
        try: shutil.copyfile(p, q + ".tmp"); os.replace(q + ".tmp", q)
        except Exception as e: print("⚠️ chep Drive loi", e, flush=True)

def chep():
    while True:
        time.sleep(300)
        for p in list(CUR): chep_mot(p)
threading.Thread(target=chep, daemon=True).start()

def chay(cmd, log, cwd):
    t0 = time.time()
    with open(log, "a") as f:
        q = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, env=ENV, start_new_session=True)
        while q.poll() is None:
            time.sleep(120)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"{time.strftime('%H:%M', time.gmtime(time.time()+7*3600))} VN · {(time.time()-T0)/3600:5.2f} h · {os.path.basename(log)} · {L[-1][:110] if L else 'chua co dong'}", flush=True)
    print(f"--- {os.path.basename(log)} · ma thoat {q.returncode} · {(time.time()-t0)/3600:.2f} h", flush=True)
    if q.returncode != 0:
        print(open(log, errors="ignore").read()[-3000:], flush=True)
    return q.returncode

# kiem bo cham: 10 cau S1 phai trung tep tho cu
S1R = {(x["episode_id"], x["step_id"]): x for x in map(json.loads, open(f"{KIT}/raw_s1.jsonl"))}
ST = f"{W}/kiem_cham.json"; STR = ST.replace(".json", "_raw.jsonl")
if os.path.exists(STR): os.remove(STR)
assert chay([sys.executable, "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground", "--preds", f"{KIT}/preds_s1_seed101.jsonl", "--out", ST, "--n", "10"], f"{W}/kiem_cham.log", WS) == 0, "DUNG: bo cham loi"
X = [json.loads(l) for l in open(STR)]
trung = sum(x.get("pred_xy") == S1R[(x["episode_id"], x["step_id"])].get("pred_xy") for x in X)
print(f"[kiem bo cham] {trung}/{len(X)} buoc trung toa do tep tho S1", flush=True)
assert len(X) >= 5 and trung >= len(X) - 1, "DUNG: bo cham tren may nay khong tai lap ban cu"

# hoa S1, kiem hoa
if not os.path.exists(f"{MERGED}/config.json"):
    r = subprocess.run([sys.executable, "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED], cwd=W, capture_output=True, text=True, env=ENV)
    print((r.stdout + r.stderr)[-800:], flush=True); assert r.returncode == 0, "DUNG: hoa loi"
kh = f"{W}/kiem_hoa_test.jsonl"
if os.path.exists(kh): os.remove(kh)
assert chay([sys.executable, "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--s1", f"{KIT}/preds_s1_seed101.jsonl", "--n", "20", "--out", kh] + TA_ARGS, f"{W}/kiem_hoa_test.log", W) == 0, "DUNG: kiem hoa loi"
dong = [l for l in open(f"{W}/kiem_hoa_test.log") if "[kiểm hoà]" in l]; print(dong, flush=True)
assert dong and int(dong[-1].split("]")[1].split("/")[0]) >= 16, "DUNG: kiem hoa duoi 16/20"

KQ = {}
for R, TEN, DIR, _ in LUOT:
    print(f"\n===== {TEN} =====", flush=True)
    PRED, SC = f"{W}/pred_{R}_test.jsonl", f"{W}/score_{R}_test.json"
    RAW = SC.replace(".json", "_raw.jsonl"); GIU = RAW + ".chua_lai.json"
    for p in (PRED, RAW, GIU):                                  # khoi phuc tu Drive, giu ban dai hon
        q = f"{DR}/{os.path.basename(p)}"
        if os.path.exists(q) and dem(q) > dem(p):
            shutil.copyfile(q, p); print("khoi phuc", os.path.basename(p), dem(p), "dong", flush=True)
    CUR[:] = [PRED, RAW, GIU]
    if dem(RAW) < 4463:
        if dem(PRED) < 4463:
            assert chay([sys.executable, "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--ckpt", f"{KIT}/{DIR}", "--out", PRED] + TA_ARGS, f"{W}/gen_{R}.log", W) == 0, f"DUNG: sinh cau {R} loi"
            print([l.strip() for l in open(f"{W}/gen_{R}.log") if "[điểm lưu]" in l], flush=True)
        n = dem(PRED); assert n == 4463, f"DUNG: {R} co {n} cau"
        r = subprocess.run([sys.executable, "tai_dung_cham.py", "gieo", "--preds", PRED, "--raw", RAW, "--nguon"] + NGUON, cwd=W, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:], flush=True); assert r.returncode == 0, "DUNG: gieo loi"
        chep_mot(RAW); chep_mot(GIU)
        assert chay([sys.executable, "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground", "--preds", PRED, "--out", SC], f"{W}/cham_{R}.log", WS) == 0, f"DUNG: cham {R} loi"
    r = subprocess.run([sys.executable, "tai_dung_cham.py", "kiem", "--raw", RAW, "--nguon"] + NGUON, cwd=W, capture_output=True, text=True)
    print(r.stdout[-400:], r.stderr[-400:], flush=True); assert r.returncode == 0, "DUNG: bo cham khong tai lap, khong duoc dung so"
    assert dem(RAW) == 4463, dem(RAW)
    DOC = f"{W}/doc_{R}.json"
    assert sh(f"cd {W} && {sys.executable} doc_luot.py --harness {WS}/harness --s1 {KIT}/raw_s1.jsonl --ck {KIT}/raw_ck500.jsonl --ten '{TEN}' --raw {RAW} --out {DOC}", 400) == 0, "DUNG: doc ket qua loi"
    D = json.load(open(DOC)); KQ[TEN] = D
    print(f"\n----- {TEN} -----")
    for k, v in D["diem"].items(): print(f"{k:20s} exec {v['exec']:.2f} · dung loai {v['dung_loai']:.2f} · so tu {v['so_tu']:.2f}")
    for k, nhan in (("x_tru_s1", f"{TEN} − S1"), ("ck500_tru_x", f"ck500 − {TEN}")):
        v = D[k]; print(f"{nhan:30s} {v['delta']:+.2f} [{v['lo']:+.2f}; {v['hi']:+.2f}] · hon {v['hon']} kem {v['kem']} · p {v['p']:.2g}")
    for k, v in D["phan_ra_so_s1"].items(): print(k, "so S1: loai", v["loai"]["rong"], "· phan tu", v["phan_tu"]["rong"], f"(p loai {v['loai']['p']:.2g}, p phan tu {v['phan_tu']['p']:.2g})")
    print("cau trung:", D["cau_trung"], flush=True)
    with zipfile.ZipFile(f"{W}/{R}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in [PRED, RAW, GIU, SC, DOC] + glob.glob(f"{W}/gen_{R}.log") + glob.glob(f"{W}/cham_{R}.log") + glob.glob(f"{W}/kiem_*.log"):
            if os.path.exists(f): z.write(f, os.path.basename(f))
    shutil.copyfile(f"{W}/{R}.zip", f"{DR}/{R}.zip")
    print(f"✅ {TEN} xong · {(time.time()-T0)/3600:.2f} h · da luu {DR}/{R}.zip", flush=True)
    NGUON = NGUON + [RAW]                                       # luot sau dung lai duoc ket qua cham cua luot nay
print("✅ het cac luot", flush=True)

## Sau khi xong
Tải các tệp `.zip` trong `MyDrive/thesis/luoi_test_colab/` về rồi báo tôi, tôi sẽ đặt vào `runs/luoi/test_2_11/` và ghi vào `report/320`. Độ đo văn bản (SPICE, CIDEr-D...) và ba tiêu chí còn lại tính sau bằng kernel CPU trên Kaggle.